# 06 — Recovery Comparisons
## Trade Winds & Headwinds: USVI, 2000–2025

Indexed comparisons using **2016 = 100** as the pre-hurricane baseline.

Series indexed:
- Total visitor arrivals
- Overnight tourists
- Cruise passengers
- Hotel occupancy rate
- L&H employment (FRED SA annual average)
- Hotel rooms (USVIBER survey)
- A&F real value added (BEA)
- VIIS NP visitation

**Method:** Each series divided by its 2016 value × 100. Series with no 2016 observation are excluded.  
**No single composite score.** Series are compared visually; divergence is the finding.

In [1]:
import sys, importlib
from pathlib import Path
import os

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.cleaning as _c
import src.charts as _ch

importlib.reload(_c)
importlib.reload(_ch)

from src.cleaning import (
    load_usviber_tour24,
    load_fred_employment,
    load_bea,
    load_nps_annual,
)
from src.charts import PALETTE

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")

Ready.


In [2]:
tour = load_usviber_tour24()
fred = load_fred_employment()
bea_real = load_bea("2.3")
viis_ann = load_nps_annual("VIIS")

# FRED annual average (SA)
fred_ann = (
    fred.groupby("year")["emp_sa"]
    .mean()
    .reset_index()
    .rename(columns={"emp_sa": "emp_sa_avg"})
)

# BEA real A&F value added
af_real = (
    bea_real[bea_real["label"].str.contains("ccommodation", case=False, na=False)]
    .drop_duplicates(subset="year")[["year", "value"]]
    .rename(columns={"value": "af_real"})
)

print("Data loaded. Base year: 2016.")

Data loaded. Base year: 2016.


---
## 1. Build the Index Table

In [3]:
BASE_YEAR = 2016


def _index(df, year_col, val_col, base_year=BASE_YEAR):
    """Return a copy with an added 'idx' column (base_year = 100)."""
    base = df.loc[df[year_col] == base_year, val_col]
    if base.empty or pd.isna(base.values[0]) or base.values[0] == 0:
        return None
    base_val = base.values[0]
    out = df[[year_col, val_col]].copy()
    out["idx"] = out[val_col] / base_val * 100
    return out


series = {
    "Total arrivals": _index(tour, "year", "total_visitors"),
    "Overnight tourists": _index(tour, "year", "tourists"),
    "Cruise passengers": _index(tour, "year", "cruise_passengers"),
    "Hotel occupancy (pp)": _index(tour, "year", "occupancy_rate_pct"),
    "Hotel rooms": _index(tour, "year", "hotel_rooms"),
    "L&H employment (FRED)": _index(fred_ann, "year", "emp_sa_avg"),
    "A&F real value added": _index(af_real, "year", "af_real"),
    "VIIS NP visits": _index(
        viis_ann.query("year >= 2000"), "year", "recreation_visitors"
    ),
}

# Build a combined wide table
all_years = sorted(
    set().union(*[set(v["year"]) for v in series.values() if v is not None])
)
idx_wide = pd.DataFrame({"year": all_years})
for name, df in series.items():
    if df is not None:
        idx_wide = idx_wide.merge(
            df[["year", "idx"]].rename(columns={"idx": name}), on="year", how="left"
        )

print(f"Index table: {len(idx_wide)} years × {len(idx_wide.columns)-1} series")
key_years = [
    2000,
    2010,
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
]
print(idx_wide[idx_wide["year"].isin(key_years)].set_index("year").round(1).to_string())

Index table: 37 years × 8 series
      Total arrivals  Overnight tourists  Cruise passengers  Hotel occupancy (pp)  Hotel rooms  L&H employment (FRED)  A&F real value added  VIIS NP visits
year                                                                                                                                                       
2000            97.4                81.9               99.5                 101.8        103.4                   88.4                   NaN           171.1
2010           104.4                86.3              104.6                 100.7        101.5                   94.0                 104.0           120.0
2015           102.7                96.1              105.7                 100.7        102.2                   96.2                 102.3           106.6
2016           100.0               100.0              100.0                 100.0        100.0                  100.0                 100.0           100.0
2017            76.0           

---
## 2. Full Recovery Index Chart

In [4]:
series_styles = [
    ("Total arrivals", PALETTE["total"], "solid", 2.5),
    ("Overnight tourists", PALETTE["tourists"], "dash", 2),
    ("Cruise passengers", PALETTE["cruise"], "dot", 2),
    ("Hotel rooms", PALETTE["rooms"], "solid", 2),
    ("L&H employment (FRED)", PALETTE["employment"], "dash", 2),
    ("A&F real value added", PALETTE["value_added"], "longdash", 2),
    ("VIIS NP visits", PALETTE["nps_viis"], "dashdot", 1.5),
]

fig = go.Figure()
for name, color, dash, width in series_styles:
    if name not in idx_wide.columns:
        continue
    sub = idx_wide[["year", name]].dropna()
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub[name],
            mode="lines",
            name=name,
            line=dict(color=color, dash=dash, width=width),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#ccc", width=1, dash="dot"),
    annotation_text="2016 = 100",
    annotation_position="right",
    annotation_font=dict(size=10, color="#999"),
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )

fig.update_layout(
    title=dict(
        text="USVI Tourism Economy — Recovery Index (2016 = 100)<br>"
        '<span style="font-size:13px;color:#555">Multiple indicators indexed to pre-hurricane baseline. '
        "Divergence indicates uneven recovery across dimensions.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Index (2016 = 100)",
    height=540,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(r=180),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.07,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, BEA, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 3. Hurricane Shock and Recovery

In [5]:
# Focused view: 2014–2024
focus = idx_wide.query("2014 <= year <= 2025").copy()

fig = go.Figure()
for name, color, dash, width in series_styles:
    if name not in focus.columns:
        continue
    sub = focus[["year", name]].dropna()
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub[name],
            mode="lines+markers",
            name=name,
            line=dict(color=color, dash=dash, width=width),
            marker=dict(size=5),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#ccc", width=1, dash="dot"),
    annotation_text="2016 = 100",
    annotation_position="right",
    annotation_font=dict(size=10, color="#999"),
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )

fig.update_layout(
    title=dict(
        text="Recovery Index — Shock Period and Aftermath (2014–2025)<br>"
        '<span style="font-size:13px;color:#555">2016 = 100. Shows both hurricane and COVID shocks and subsequent recovery.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Index (2016 = 100)",
    height=520,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(r=180),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.07,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, BEA, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 4. Recovery Scorecard

In [6]:
# For each series: value at key comparison years
comparison_years = [2016, 2017, 2019, 2020, 2022, 2024, 2025]

scorecard = idx_wide[idx_wide["year"].isin(comparison_years)].set_index("year").T
scorecard.columns = [str(y) for y in scorecard.columns]

print("=== Recovery Index Scorecard (2016 = 100) ===\n")
print(scorecard.round(1).to_string())

print("\n=== Change from 2016 to most recent year ===")
for name in scorecard.index:
    row = scorecard.loc[name]
    latest_col = [
        c for c in ["2025", "2024", "2022"] if c in row.index and pd.notna(row[c])
    ]
    if not latest_col:
        continue
    latest = row[latest_col[0]]
    val_2016 = row.get("2016", 100.0)
    status = "✓ above baseline" if latest >= 100 else "✗ below baseline"
    print(f"  {name:35s} {latest_col[0]}: {latest:6.1f}  {status}")

=== Recovery Index Scorecard (2016 = 100) ===

                       2016  2017 2019 2020  2022  2024  2025
Total arrivals        100.0  76.0 80.7 30.7  69.4  97.0   NaN
Overnight tourists    100.0  80.1 62.6 20.3  60.3  56.9   NaN
Cruise passengers     100.0  74.1 80.7 24.8  56.5  90.0   NaN
Hotel occupancy (pp)  100.0  77.4 70.0 65.3 112.7 116.9   NaN
Hotel rooms           100.0 100.0 79.9 66.9  69.4  81.9   NaN
L&H employment (FRED) 100.0  89.9 73.0 63.6  81.5  90.8  84.4
A&F real value added  100.0  87.9 64.0 46.7   NaN   NaN   NaN
VIIS NP visits        100.0  74.0 32.4 40.7  47.8 102.8 114.5

=== Change from 2016 to most recent year ===
  Total arrivals                      2024:   97.0  ✗ below baseline
  Overnight tourists                  2024:   56.9  ✗ below baseline
  Cruise passengers                   2024:   90.0  ✗ below baseline
  Hotel occupancy (pp)                2024:  116.9  ✓ above baseline
  Hotel rooms                         2024:   81.9  ✗ below baseline
  L&

In [7]:
# Bar chart of most-recent index values vs. 2016 baseline
latest_vals = {}
for name in scorecard.index:
    row = scorecard.loc[name]
    for yr in ["2025", "2024", "2022", "2021"]:
        if yr in row.index and pd.notna(row[yr]):
            latest_vals[name] = (yr, row[yr])
            break

names = list(latest_vals.keys())
labels = [f"{n} ({latest_vals[n][0]})" for n in names]
values = [latest_vals[n][1] for n in names]
bar_colors = [PALETTE["rooms"] if v >= 100 else PALETTE["cruise"] for v in values]

fig = go.Figure(
    go.Bar(
        x=values,
        y=labels,
        orientation="h",
        marker_color=bar_colors,
        text=[f"{v:.0f}" for v in values],
        textposition="outside",
    )
)
fig.add_vline(
    x=100,
    line=dict(color="#444", width=1.5),
    annotation_text="2016 = 100",
    annotation_position="top",
    annotation_font=dict(size=10),
)
fig.update_layout(
    title=dict(
        text="Most Recent Index vs. 2016 Baseline<br>"
        '<span style="font-size:13px;color:#555">Blue = at or above 2016; orange = below 2016.</span>',
        font=dict(size=17),
    ),
    xaxis_title="Index (2016 = 100)",
    xaxis_range=[0, max(values) * 1.15],
    height=420,
    margin=dict(l=240),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.1,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, BEA, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 5. COVID Shock Depth and Recovery

In [8]:
# Using 2019 as the COVID baseline (last pre-pandemic year)
BASE_COVID = 2019

series_covid = {
    "Total arrivals": _index(tour, "year", "total_visitors", BASE_COVID),
    "Overnight tourists": _index(tour, "year", "tourists", BASE_COVID),
    "Cruise passengers": _index(tour, "year", "cruise_passengers", BASE_COVID),
    "Hotel rooms": _index(tour, "year", "hotel_rooms", BASE_COVID),
    "L&H employment (FRED)": _index(fred_ann, "year", "emp_sa_avg", BASE_COVID),
    "VIIS NP visits": _index(
        viis_ann.query("year >= 2015"), "year", "recreation_visitors", BASE_COVID
    ),
}

fig = go.Figure()
for name, color, dash, width in series_styles:
    if name not in series_covid or series_covid[name] is None:
        continue
    sub = series_covid[name].query("year >= 2017").dropna()
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["idx"],
            mode="lines+markers",
            name=name,
            line=dict(color=color, dash=dash, width=width),
            marker=dict(size=5),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#ccc", width=1, dash="dot"),
    annotation_text="2019 = 100",
    annotation_position="right",
    annotation_font=dict(size=10, color="#999"),
)
fig.add_vline(
    x=2020,
    line=dict(color="#555", width=1.5, dash="dash"),
    annotation_text="COVID-19",
    annotation_position="top",
    annotation_font=dict(size=10, color="#444"),
)

fig.update_layout(
    title=dict(
        text="COVID Shock and Recovery (2019 = 100)<br>"
        '<span style="font-size:13px;color:#555">Indexed to last pre-pandemic year. Note: hotel room count already post-hurricane in 2019.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Index (2019 = 100)",
    height=500,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(r=180),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.07,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 6. Preliminary Observations

*(These are analytical observations from the data, not conclusions for publication. They inform what deserves deeper investigation.)*

Run the scorecard cell and examine the index table before forming any conclusions. Key questions to explore in the Field Note phase:

1. **Did total visitor arrivals recover to 2016 levels by 2024?** Check the scorecard.
2. **Did overnight tourists recover at the same rate as cruise passengers?** Look for divergence in the composition.
3. **Did employment recover in step with visitor volume?** The Census 2022 accommodation employment (1,650 jobs) vs. 2012 (3,876) suggests structural change beyond what visit counts show.
4. **Did room inventory recover?** The room count trajectory (USVIBER vs. Census) tells different stories — investigate which is more reliable.
5. **Is 2024 occupancy rate (66.3%) telling a different story than absolute visitor counts?** High occupancy on a smaller room base may represent profitable operation at reduced scale, not full recovery.
6. **Why is VIIS NP visitation so different from territory-wide arrivals?** The park draws overnight tourists, not cruise passengers — the VIIS trajectory should track overnight tourist arrivals more closely than total arrivals.